# Evaluating Conversion Uplift from Marketing Ads vs PSA: An A/B Test Analysis

## Business Question

Does replacing PSA content with a marketing ad result in a statistically and practically significant increase in conversion rate, justifying a full rollout?

In [42]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [43]:
df = pd.read_csv("marketing_AB.csv")

## Data Understanding

### Basic Overview
The dataset contains user-level observations from an A/B test comparing a marketing ad (treatment) against a PSA (control).

Key fields:
- `user_id`: Unique identifier for each user
- `test_group`: Indicates assignment to control (PSA) or treatment (ad)
- `converted`: Binary indicator (1 = conversion, 0 = no conversion)
- `timestamp`: Time of user interaction

Each row represents a single user exposure to either variant.

In [44]:
df.head()

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [45]:
df.shape

(588101, 7)

In [46]:
#data structure overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Unnamed: 0     588101 non-null  int64 
 1   user id        588101 non-null  int64 
 2   test group     588101 non-null  object
 3   converted      588101 non-null  bool  
 4   total ads      588101 non-null  int64 
 5   most ads day   588101 non-null  object
 6   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(4), object(2)
memory usage: 27.5+ MB


### Data Quality Checks

- Verifying no missing values in key fields (`test_group`, `converted`)
- Ensured each user appears only once (no duplicate exposures)
- Remove unnecessary columns
- Column cleanup, renaming and data type correction

In [47]:
#Missing values
df.isnull().sum()

Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

In [48]:
# Check duplicates
df.duplicated().sum()

np.int64(0)

In [49]:
# Check unique users
df['user id'].nunique()

588101

### Experiment Sanity Checks

In [50]:
df['test group'].value_counts(normalize=True)

test group
ad     0.96
psa    0.04
Name: proportion, dtype: float64

The 96/4 split is atypical for a standard A/B test and may indicate an unequal allocation design, a data collection issue, or a sample ratio mismatch. Since the dataset provides no documentation on intended allocation, this is flagged as a limitation of the analysis.

---

## Data Preparation

In [51]:
# Remove duplicates
df = df.drop_duplicates(subset='user id')

In [52]:
#Column cleanup
df.drop(columns='Unnamed: 0', inplace=True)

In [53]:
#Column renaming
df = df.rename(columns={
    'user id': 'user_id',
    'test group': 'test_group',
    'total ads': 'total_ads',
    'most ads day': 'most_ads_day',
    'most ads hour': 'most_ads_hour'
})

#Data type correction
df['test_group'] = df['test_group'].astype('category')
df["converted"] = df["converted"].astype(int)
df['most_ads_day'] = df['most_ads_day'].astype("category")
df['most_ads_hour'] = df['most_ads_hour'].astype("category")

- No missing values were found in key columns (test_group, converted), ensuring reliability of group assignment and outcome measurement.

- No duplicate rows were detected, and each user appears only once, confirming no repeated exposure to variants.

- Column "Unnamed: 0" is not needed and so is removed.

Overall, the dataset passes key validity checks, including balanced group assignment, complete data, and consistent user-level observations. This supports reliable A/B test analysis.

---

## Problem Statement
Given the imbalance in group allocation, the objective is to evaluate whether the observed difference in conversion rates between ad and psa groups is statistically significant.


### Hypothesis

H₀ (Null Hypothesis): The conversion rate of the ad group is less than or equal to the psa group.

H₁ (Alternative Hypothesis): The conversion rate of the ad group is greater than the psa group.

### Exploratory Data Analysis (EDA)
 #### Conversion Rate comparison

In [64]:
df.groupby('test_group')['converted'].mean()

test_group
ad     0.025547
psa    0.017854
Name: converted, dtype: float64

The ad group shows a higher conversion rate compared to the psa group.
This suggests a potential positive effect of the ad intervention.

#### Distribution Check

In [55]:
pd.crosstab(df['test_group'],df['most_ads_day'],normalize='index')

most_ads_day,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
test_group,,,,,,,
ad,0.157295,0.148024,0.139577,0.145830,0.140064,0.132085,0.137126
psa,0.161665,0.148869,0.121493,0.130037,0.166001,0.123576,0.148359


The distribution across days is broadly similar between groups, with only minor variations observed. These differences are not substantial enough to indicate strong allocation bias.

---

### Statistical Analysis

In [56]:
#Effect Size
cr = df.groupby('test_group')['converted'].mean()
effect_size = cr['ad'] - cr['psa']
print(effect_size)

0.007692453192201517


The conversion rate difference between the ad and psa groups is approximately 0.77 percentage points.

This represents the observed effect size, indicating that the ad group performs better in terms of conversion

In [57]:
# performing Z test
conversions = df.groupby('test_group')['converted'].sum()
users = df.groupby('test_group')['converted'].count()

count = [conversions['ad'], conversions['psa']]
nobs = [users['ad'], users['psa']]

z_stat, p_value = proportions_ztest(count, nobs, alternative='larger')
print(p_value)

8.526403580779863e-14


The p-value is extremely small (<< 0.05), indicating strong statistical evidence against the null hypothesis.

Therefore, we reject the null hypothesis and conclude that :
The conversion rate of the ad group is statistically significantly higher than that of the psa group.

In [58]:
count1 = conversions['ad']
nobs1 = users['ad']

count2 = conversions['psa']
nobs2 = users['psa']

low, high = confint_proportions_2indep(
    count1, nobs1,
    count2, nobs2,
    method='wald'
)

print("Confidence Interval:", low, "to", high)

Confidence Interval: 0.005950932431611005 to 0.00943397395279203


The 95% confidence interval for the difference in conversion rates lies between 0.60% and 0.94%.

Since the interval does not include 0, it confirms that the observed difference is statistically significant.

This indicates a consistent and positive lift in conversion for the ad group.

In [59]:
# Observed conversion rates
cr_ad = conversions['ad'] / users['ad']
cr_psa = conversions['psa'] / users['psa']

effect = proportion_effectsize(cr_ad, cr_psa)

analysis = NormalIndPower()
power = analysis.solve_power(
    effect_size=effect,
    nobs1=users['ad'],
    ratio=users['psa'] / users['ad'],
    alpha=0.05,
    alternative='larger'
)

print(f"Observed power: {power}")

Observed power: 0.9999999998693799


The observed statistical power is very high (close to 1), indicating that the experiment is well-powered to detect the observed effect and reducing the likelihood of false negatives.

In [60]:
# Run logistic regression
model = smf.logit(
    formula='converted ~ test_group + most_ads_day + most_ads_hour',
    data=df
).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.116987
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:               588101
Model:                          Logit   Df Residuals:                   588070
Method:                           MLE   Df Model:                           30
Date:                Wed, 06 May 2026   Pseudo R-squ.:                0.006740
Time:                        23:56:56   Log-Likelihood:                -68800.
converged:                       True   LL-Null:                       -69267.
Covariance Type:            nonrobust   LLR p-value:                5.006e-177
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -4.0975      0.102    -40.119      0.000      -4.

Logistic regression was used to control for temporal factors such as day and hour of exposure.

The coefficient for test_group[T.psa] is negative and statistically significant, indicating a lower likelihood of conversion compared to the ad group.

This suggests that the observed effect persists even after adjusting for these factors.

In [63]:
#Odds Ratio
odds_ratio = np.exp(model.params['test_group[T.psa]'])
print("Odds Ratio:", odds_ratio)

Odds Ratio: 0.6933500663702414


The odds ratio is approximately 0.69, meaning that users in the psa group are about 31% less likely to convert compared to users in the ad group.

This provides an interpretable measure of effect size, reinforcing the strength of the observed difference.



## Interpretation of Results

The ad group consistently demonstrates higher conversion rates compared to the psa group across all analyses.

The difference is statistically significant, as supported by the hypothesis test and confidence interval.

Despite the imbalance in group allocation, the large sample size and high statistical power suggest that the results are reliable.

Distribution checks indicate no major allocation bias across time, and regression analysis shows that the effect remains significant after controlling for temporal factors.

---

## Final Conclusion

The analysis provides strong evidence that the ad intervention leads to a statistically significant increase in conversion rates compared to the psa group.

While the magnitude of the lift is modest, it is consistent and robust across multiple analytical approaches.

**Recommendation: The ad version should be considered for deployment.**

---

## Extended Analysis: When Should Ads Be Shown?

While the primary analysis confirms the ad variant drives higher conversion, 
a secondary question is equally relevant for campaign optimization: 
**are there specific days or hours where the ad effect is strongest?**

In [ ]:
day_order = [  'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday' ][::-1]

df['most_ads_day'] = pd.Categorical(df['most_ads_day'], categories=day_order, ordered=True)

ad_df = df[df['test_group'] == 'ad']

pivot = ad_df.pivot_table(
    values='converted',
    index='most_ads_day',
    columns='most_ads_hour',
    aggfunc='mean'
)


import plotly.graph_objects as go

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns,
    y=pivot.index,
    colorscale="RdBu_r",
    hovertemplate=
        "Day: %{y}<br>" +
        "Hour: %{x}<br>" +
        "Conversion: %{z:.2%}<extra></extra>"
))

fig.update_layout(
    title=dict(
        text="<b>Conversion Rate by Day and Hour</b>",
        x=0.5
    ),

    xaxis=dict(
        title="Hour",
        tickmode='array',
        tickvals=pivot.columns   # show every hour
    ),

    yaxis=dict(
        title="Day"
    ),

    template="plotly_white",
    width=1300,
    height=600
)

fig.show()

**Recommendation: Prioritize ad delivery during Saturday early morning (5–6 AM) 
and evening slots (7–9 PM) across weekends. Reduce or avoid ad spend during 
2–4 AM across all days, where conversion is consistently weakest.**

---
